In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga input at output ng Primitive

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Available na ang beta release ng bagong execution model. Ang directed execution model ay nagbibigay ng mas maraming flexibility kapag kina-customize ang iyong error mitigation workflow. Tingnan ang gabay na [Directed execution model](/guides/directed-execution-model) para sa karagdagang impormasyon.


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa page na ito ay binuo gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang page na ito ay nagbibigay ng pangkalahatang-ideya ng mga input at output ng Qiskit Runtime primitives na nagpapatupad ng mga workload sa IBM Quantum&reg; compute resources. Ang mga primitive na ito ay nagbibigay sa iyo ng kakayahang mahusay na tukuyin ang mga vectorized workload sa pamamagitan ng isang data structure na kilala bilang **Primitive Unified Bloc (PUB)**. Ang mga PUB na ito ang pangunahing yunit ng trabaho na kailangang isagawa ng isang QPU para sa mga workload na ito. Ginagamit ang mga ito bilang mga input sa pamamaraan ng [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) para sa mga Sampler at Estimator primitive, na nagpapatupad ng tinukoy na workload bilang isang job. Pagkatapos, kapag nakumpleto na ang job, ang mga resulta ay ibinalik sa isang format na depende sa parehong mga PUB na ginamit at sa mga runtime option na tinukoy mula sa mga Sampler o Estimator primitive.
<span id="pubs"></span>
## Pangkalahatang-ideya ng mga PUB
Kapag tinatawagan ang pamamaraan ng [`run()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2#run) ng isang primitive, ang pangunahing argument na kailangan ay isang `list` ng isa o higit pang mga tuple — isa para sa bawat Circuit na pinapatakbo ng primitive. Ang bawat isa sa mga tuple na ito ay itinuturing na isang PUB, at ang mga kinakailangang elemento ng bawat tuple sa listahan ay nakasalalay sa primitive na ginamit. Ang data na ibinibigay sa mga tuple na ito ay maaari ring ayusin sa iba't ibang hugis upang magbigay ng flexibility sa isang workload sa pamamagitan ng broadcasting — ang mga patakaran nito ay inilarawan sa isang [sumusunod na seksyon](#broadcasting-rules).

### Estimator PUB
Para sa Estimator primitive, ang format ng PUB ay dapat maglaman ng hindi hihigit sa apat na halaga:
- Isang solong `QuantumCircuit`, na maaaring maglaman ng isa o higit pang mga [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) na object
- Isang listahan ng isa o higit pang mga observable, na nagtutukoy ng mga inaasahang halaga na tatantiyahin, na nakaayos sa isang array (halimbawa, isang solong observable na kinakatawan bilang isang 0-d na array, isang listahan ng mga observable bilang isang 1-d na array, at iba pa). Ang data ay maaaring nasa alinman sa `ObservablesArrayLike` na format tulad ng `Pauli`, `SparsePauliOp`, `PauliList`, o `str`.
   > **Note:** Kung mayroon kang dalawang commuting observable sa iba't ibang PUB ngunit may parehong Circuit, hindi sila tatantiyahin gamit ang parehong sukat. Ang bawat PUB ay kumakatawan sa ibang batayan para sa pagsukat, kaya naman, kinakailangan ang magkahiwalay na mga pagsukat para sa bawat PUB. Upang matiyak na ang mga commuting observable ay natantiya gamit ang parehong pagsukat, dapat silang ipangkat sa loob ng parehong PUB.
- Isang koleksyon ng mga halaga ng parameter para i-bind ang Circuit. Maaari itong tukuyin bilang isang solong array-like na object kung saan ang huling index ay nasa mga `Parameter` na object ng Circuit, o maaaring alisin (o katumbas nito, itakda sa `None`) kung ang Circuit ay walang mga `Parameter` na object.
- (Opsyonal) isang target na katumpakan para sa mga inaasahang halaga na tatantiyahin

### Sampler PUB
Para sa Sampler primitive, ang format ng PUB tuple ay naglalaman ng hindi hihigit sa tatlong halaga:
- Isang solong `QuantumCircuit`, na maaaring maglaman ng isa o higit pang mga [`Parameter`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.Parameter) na object
   *Tandaan: Ang mga Circuit na ito ay dapat ding maglaman ng mga tagubilin sa pagsukat para sa bawat Qubit na susukatin.*
- Isang koleksyon ng mga halaga ng parameter para i-bind ang Circuit laban sa $\theta_k$ (kailangan lamang kung may mga `Parameter` na object na ginagamit na dapat i-bind sa runtime)
- (Opsyonal) bilang ng mga shot para sukatin ang Circuit
---

Ang sumusunod na code ay nagpapakita ng isang halimbawa ng set ng mga vectorized input sa `Estimator` primitive at pinapatakbo ang mga ito sa isang IBM&reg; backend bilang isang solong `RuntimeJobV2` na object.

In [1]:
from qiskit.circuit import (
    Parameter,
    QuantumCircuit,
    ClassicalRegister,
    QuantumRegister,
)
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.containers import BitArray

from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as Estimator,
    SamplerV2 as Sampler,
)

import numpy as np

# Instantiate runtime service and get
# the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Define a circuit with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.rz(Parameter("b"), 0)
circuit.cx(0, 1)
circuit.h(0)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)
layout = transpiled_circuit.layout

# Now define a sweep over parameter values, the last axis of dimension 2 is
# for the two parameters "a" and "b"
params = np.vstack(
    [
        np.linspace(-np.pi, np.pi, 100),
        np.linspace(-4 * np.pi, 4 * np.pi, 100),
    ]
).T

# Define three observables. The inner length-1 lists cause this array of
# observables to have shape (3, 1), rather than shape (3,) if they were
# omitted.
observables = [
    [SparsePauliOp(["XX", "IY"], [0.5, 0.5])],
    [SparsePauliOp("XX")],
    [SparsePauliOp("IY")],
]
# Apply the same layout as the transpiled circuit.
observables = [
    [observable.apply_layout(layout) for observable in observable_set]
    for observable_set in observables
]

# Estimate the expectation value for all 300 combinations of observables
# and parameter values, where the pub result will have shape (3, 100).
#
# This shape is due to our array of parameter bindings having shape
# (100, 2), combined with our array of observables having shape (3, 1).
estimator_pub = (transpiled_circuit, observables, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
estimator = Estimator(mode=backend)
job = estimator.run([estimator_pub])
result = job.result()

### Mga patakaran sa broadcasting
Ang mga PUB ay nag-aahon ng mga elemento mula sa maraming array (mga observable at mga halaga ng parameter) sa pamamagitan ng pagsunod sa parehong mga patakaran sa broadcasting ng NumPy. Ang seksyong ito ay maikling buod ng mga patakarang iyon. Para sa detalyadong paliwanag, tingnan ang [dokumentasyon ng broadcasting rules ng NumPy](https://numpy.org/doc/stable/user/basics.broadcasting.html).

Mga patakaran:

* Ang mga input array ay hindi kailangang magkaroon ng parehong bilang ng mga dimensyon.
  * Ang resulting array ay magkakaroon ng parehong bilang ng mga dimensyon katulad ng input array na may pinakamalaking dimensyon.
  * Ang laki ng bawat dimensyon ay ang pinakamalaking laki ng katumbas na dimensyon.
  * Ang mga nawawalang dimensyon ay ipinapalagay na may laki na isa.
* Ang mga paghahambing ng hugis ay nagsisimula sa pinaka-kanang dimensyon at nagpapatuloy sa kaliwa.
* Ang dalawang dimensyon ay tugma kung ang kanilang mga laki ay pantay o kung isa sa kanila ay 1.

Mga halimbawa ng mga pares ng array na nag-broadcast:

In [2]:
# Broadcast single observable
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = SparsePauliOp("ZZZ")  # shape ()
# >> pub result has shape (5,)

# Zip
parameter_values = np.random.uniform(size=(5,))  # shape (5,)
observables = [
    SparsePauliOp(pauli) for pauli in ["III", "XXX", "YYY", "ZZZ", "XYZ"]
]  # shape (5,)
# >> pub result has shape (5,)

# Outer/Product
parameter_values = np.random.uniform(size=(1, 6))  # shape (1, 6)
observables = [
    [SparsePauliOp(pauli)] for pauli in ["III", "XXX", "YYY", "ZZZ"]
]  # shape (4, 1)
# >> pub result has shape (4, 6)

# Standard nd generalization
parameter_values = np.random.uniform(size=(3, 6))  # shape (3, 6)
observables = [
    [
        [SparsePauliOp(["XII"])],
        [SparsePauliOp(["IXI"])],
        [SparsePauliOp(["IIX"])],
    ],
    [
        [SparsePauliOp(["ZII"])],
        [SparsePauliOp(["IZI"])],
        [SparsePauliOp(["IIZ"])],
    ],
]  # shape (2, 3, 1)
# >> pub result has shape (2, 3, 6)

Mga halimbawa ng mga pares ng array na hindi nag-broadcast:

In [3]:
print(
    f"The result of the submitted job had {len(result)} PUB and has a value:\n {result}\n"
)
print(
    f"The associated PubResult of this job has the following data bins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    "Recall that this shape is due to our array of parameter binding sets having shape (100, 2) -- where 2 is the\n\
         number of parameters in the circuit -- combined with our array of observables having shape (3, 1). \n"
)
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB and has a value:
 PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=(3, 100), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(3, 100), dtype=float64>), shape=(3, 100)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})

The associated PubResult of this job has the following data bins:
 DataBin(evs=np.ndarray(<shape=(3, 100), dtype=float64>), stds=np.ndarray(<shape=

Ang `EstimatorV2` ay nagbabalik ng isang tinatantiyahang inaasahang halaga para sa bawat elemento ng broadcasted shape.

Narito ang ilang halimbawa ng mga karaniwang pattern na ipinahayag sa mga termino ng array broadcasting. Ang kanilang kasamang visual na representasyon ay ipinapakita sa sumusunod na figure:

Ang mga set ng halaga ng parameter ay kinakatawan ng n x m na mga array, at ang mga array ng observable ay kinakatawan ng isa o higit pang single-column na mga array. Para sa bawat halimbawa sa nakaraang code, ang mga set ng halaga ng parameter ay pinagsama sa kanilang observable array upang lumikha ng mga resulting na tinatantiyahang inaasahang halaga.

 - *Halimbawa 1*: (broadcast single observable) ay may set ng halaga ng parameter na isang 5x1 na array at isang 1x1 na observables array. Ang isang item sa observables array ay pinagsama sa bawat item sa set ng halaga ng parameter upang lumikha ng isang solong 5x1 na array kung saan ang bawat item ay isang kombinasyon ng orihinal na item sa set ng halaga ng parameter kasama ang item sa observables array.

 - *Halimbawa 2*: (zip) ay may 5x1 na set ng halaga ng parameter at isang 5x1 na observables array. Ang output ay isang 5x1 na array kung saan ang bawat item ay isang kombinasyon ng ika-n na item sa set ng halaga ng parameter kasama ang ika-n na item sa observables array.

  - *Halimbawa 3*: (outer/product) ay may 1x6 na set ng halaga ng parameter at isang 4x1 na observables array. Ang kanilang kombinasyon ay nagbubunga ng isang 4x6 na array na nilikha sa pamamagitan ng pagsasama ng bawat item sa set ng halaga ng parameter sa *bawat* item sa observables array, kaya ang bawat halaga ng parameter ay nagiging isang buong column sa output.

  - *Halimbawa 4*: (Standard nd generalization) ay may 3x6 na array ng set ng halaga ng parameter at dalawang 3x1 na observables array. Pinagsama ang mga ito upang lumikha ng dalawang 3x6 na output array sa paraang katulad ng nakaraang halimbawa.

![Ang imahe na ito ay nagpapakita ng ilang visual na representasyon ng array broadcasting](../docs/images/guides/primitive-input-output/broadcasting.svg "Visual na representasyon ng broadcasting")

In [4]:
# generate a ten-qubit GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure_all()

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains one BitArray
data = result[0].data
print(f"Databin: {data}\n")

# to access the BitArray, use the key "meas", which is the default name of
# the classical register when this is added by the `measure_all` method
array = data.meas
print(f"BitArray: {array}\n")
print(f"The shape of register `meas` is {data.meas.array.shape}.\n")
print(f"The bytes in register `alpha`, shot by shot:\n{data.meas.array}\n")

Databin: DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=10>))

BitArray: BitArray(<shape=(), num_shots=4096, num_bits=10>)

The shape of register `meas` is (4096, 2).

The bytes in register `alpha`, shot by shot:
[[  3 254]
 [  0   0]
 [  3 255]
 ...
 [  0   0]
 [  3 255]
 [  0   0]]



<Admonition type="tip" title="SparsePauliOp">
Ang bawat `SparsePauliOp` ay binibilang bilang isang solong elemento sa kontekstong ito, anuman ang bilang ng mga Pauli na nasa `SparsePauliOp`. Kaya, para sa layunin ng mga patakarang ito sa broadcasting, lahat ng sumusunod na elemento ay may parehong hugis:

In [5]:
# optionally, convert away from the native BitArray format to a dictionary format
counts = data.meas.get_counts()
print(f"Counts: {counts}")

Counts: {'1111111110': 199, '0000000000': 1337, '1111111111': 1052, '1111111000': 33, '1110000000': 65, '1100100000': 2, '1100000000': 25, '0010001110': 1, '0000000011': 30, '1111111011': 58, '1111111010': 25, '0000000110': 7, '0010000001': 11, '0000000001': 179, '1110111110': 6, '1111110000': 33, '1111101111': 49, '1110111111': 40, '0000111010': 2, '0100000000': 35, '0000000010': 51, '0000100000': 31, '0110000000': 7, '0000001111': 22, '1111111100': 24, '1011111110': 5, '0001111111': 58, '0000111111': 24, '1111101110': 10, '0000010001': 5, '0000001001': 2, '0011111111': 38, '0000001000': 11, '1111100000': 34, '0111111111': 45, '0000000100': 18, '0000000101': 2, '1011111111': 11, '1110000001': 13, '1101111000': 1, '0010000000': 52, '0000010000': 17, '0000011111': 15, '1110100001': 1, '0111111110': 9, '0000000111': 19, '1101111111': 15, '1111110111': 17, '0011111110': 5, '0001101110': 1, '0111111011': 6, '0100001000': 2, '0010001111': 1, '1111011000': 1, '0000111110': 4, '0011110010': 1

Ang mga sumusunod na listahan ng mga operator, habang katumbas sa mga tuntunin ng impormasyong nilalaman, ay may iba't ibang hugis:

In [6]:
# generate a ten-qubit GHZ circuit with two classical registers
circuit = QuantumCircuit(
    qreg := QuantumRegister(10),
    alpha := ClassicalRegister(1, "alpha"),
    beta := ClassicalRegister(9, "beta"),
)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))

# append measurements with the `measure_all` method
circuit.measure([0], alpha)
circuit.measure(range(1, 10), beta)

# transpile the circuit
transpiled_circuit = pm.run(circuit)

# run the Sampler job and retrieve the results
sampler = Sampler(mode=backend)
job = sampler.run([transpiled_circuit])
result = job.result()

# the data bin contains two BitArrays, one per register, and can be accessed
# as attributes using the registers' names
data = result[0].data
print(f"BitArray for register 'alpha': {data.alpha}")
print(f"BitArray for register 'beta': {data.beta}")

BitArray for register 'alpha': BitArray(<shape=(), num_shots=4096, num_bits=1>)
BitArray for register 'beta': BitArray(<shape=(), num_shots=4096, num_bits=9>)


</Admonition>

## Pangkalahatang-ideya ng mga output ng primitive
Kapag naipadala na ang isa o higit pang PUB sa isang QPU para sa pagpapatupad at matagumpay na natapos ang isang job, ang data ay ibinabalik bilang isang [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na container object na ma-a-access sa pamamagitan ng pagtawag sa `RuntimeJobV2.result()` na method. Ang `PrimitiveResult` ay naglalaman ng isang iterable na listahan ng mga [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na object na naglalaman ng mga resulta ng pagpapatupad para sa bawat PUB. Depende sa primitive na ginamit, ang data na ito ay magiging mga expectation value at ang kanilang mga error bar sa kaso ng Estimator, o mga sample ng output ng Circuit sa kaso ng Sampler.

Ang bawat elemento ng listahang ito ay tumutugma sa bawat PUB na isinumite sa `run()` na method ng primitive (halimbawa, ang isang job na isinumite nang may 20 PUB ay magbabalik ng isang `PrimitiveResult` na object na naglalaman ng listahan ng 20 `PubResult`, isa para sa bawat PUB).

Ang bawat isa sa mga `PubResult` na object na ito ay may parehong `data` at `metadata` na attribute. Ang `data` attribute ay isang customized na [`DataBin`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) na naglalaman ng mga aktwal na halaga ng sukat, mga standard deviation, at iba pa. Ang `DataBin` na ito ay may iba't ibang attribute depende sa hugis o istruktura ng kaugnay na PUB pati na rin ang mga opsyon sa error mitigation na tinukoy ng primitive na ginamit para isumite ang job (halimbawa, [ZNE](./error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) o [PEC](./error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)). Samantala, ang `metadata` attribute ay naglalaman ng impormasyon tungkol sa mga opsyon sa runtime at error mitigation na ginamit (ipinapaliwanag sa ibaba sa seksyong [Metadata ng resulta](#result-metadata) ng pahinang ito).

Ang sumusunod ay isang visual na balangkas ng istruktura ng data ng `PrimitiveResult`:

<Tabs>
    <TabItem value="estimator" label="Output ng Estimator">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── evs
        │       │   └── List of estimated expectation values in the shape
        |       |         specified by the first pub
        │       └── stds
        │           └── List of calculated standard deviations in the
        |                 same shape as above
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       ├── evs
        |       │   └── List of estimated expectation values in the shape
        |       |        specified by the second pub
        |       └── stds
        |           └── List of calculated standard deviations in the
        |                same shape as above
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
    <TabItem value="sampler" label="Output ng Sampler">
    ```
    └── PrimitiveResult
        ├── PubResult[0]
        │   ├── metadata
        │   └── data  ## In the form of a DataBin object
        │       ├── NAME_OF_CLASSICAL_REGISTER
        │       │   └── BitArray of count data (default is 'meas')
        |       |
        │       └── NAME_OF_ANOTHER_CLASSICAL_REGISTER
        │           └── BitArray of count data (exists only if more than one
        |                 ClassicalRegister was specified in the circuit)
        ├── PubResult[1]
        |   ├── metadata
        |   └── data  ## In the form of a DataBin object
        |       └── NAME_OF_CLASSICAL_REGISTER
        |           └── BitArray of count data for second pub
        ├── ...
        ├── ...
        └── ...
    ```
    </TabItem>
</Tabs>

Sa madaling salita, ang isang job ay nagbabalik ng isang `PrimitiveResult` na object at naglalaman ng listahan ng isa o higit pang `PubResult` na object. Ang mga `PubResult` na object na ito ay nag-iimbak ng data ng sukat para sa bawat PUB na isinumite sa job.

Ang bawat `PubResult` ay may iba't ibang format at attribute batay sa uri ng primitive na ginamit para sa job. Ang mga detalye ay ipinaliwanag sa ibaba.

### Output ng Estimator
Ang bawat `PubResult` para sa Estimator primitive ay naglalaman ng kahit isang array ng mga expectation value (`PubResult.data.evs`) at mga kaugnay na standard deviation (alinman sa `PubResult.data.stds` o `PubResult.data.ensemble_standard_error` depende sa `resilience_level` na ginamit), ngunit maaaring maglaman ng higit pang data depende sa mga opsyon sa error mitigation na tinukoy.

Ang code snippet sa ibaba ay nagpapaliwanag ng format ng `PrimitiveResult` (at kaugnay na `PubResult`) para sa job na ginawa sa itaas.

In [7]:
print(f"The shape of register `alpha` is {data.alpha.array.shape}.")
print(f"The bytes in register `alpha`, shot by shot:\n{data.alpha.array}\n")

print(f"The shape of register `beta` is {data.beta.array.shape}.")
print(f"The bytes in register `beta`, shot by shot:\n{data.beta.array}\n")

# post-select the bitstrings of `beta` based on having sampled "1" in `alpha`
mask = data.alpha.array == "0b1"
ps_beta = data.beta[mask[:, 0]]
print(f"The shape of `beta` after post-selection is {ps_beta.array.shape}.")
print(f"The bytes in `beta` after post-selection:\n{ps_beta.array}")

# get a slice of `beta` to retrieve the first three bits
beta_sl_bits = data.beta.slice_bits([0, 1, 2])
print(
    f"The shape of `beta` after bit-wise slicing is {beta_sl_bits.array.shape}."
)
print(f"The bytes in `beta` after bit-wise slicing:\n{beta_sl_bits.array}\n")

# get a slice of `beta` to retrieve the bytes of the first five shots
beta_sl_shots = data.beta.slice_shots([0, 1, 2, 3, 4])
print(
    f"The shape of `beta` after shot-wise slicing is {beta_sl_shots.array.shape}."
)
print(
    f"The bytes in `beta` after shot-wise slicing:\n{beta_sl_shots.array}\n"
)

# calculate the expectation value of diagonal operators on `beta`
ops = [SparsePauliOp("ZZZZZZZZZ"), SparsePauliOp("IIIIIIIIZ")]
exp_vals = data.beta.expectation_values(ops)
for o, e in zip(ops, exp_vals):
    print(f"Exp. val. for observable `{o}` is: {e}")

# concatenate the bitstrings in `alpha` and `beta` to "merge" the results of the two
# registers
merged_results = BitArray.concatenate_bits([data.alpha, data.beta])
print(f"\nThe shape of the merged results is {merged_results.array.shape}.")
print(f"The bytes of the merged results:\n{merged_results.array}\n")

The shape of register `alpha` is (4096, 1).
The bytes in register `alpha`, shot by shot:
[[1]
 [1]
 [1]
 ...
 [0]
 [0]
 [1]]

The shape of register `beta` is (4096, 2).
The bytes in register `beta`, shot by shot:
[[  0 135]
 [  0 247]
 [  1 247]
 ...
 [  0   0]
 [  1 224]
 [  1 255]]

The shape of `beta` after post-selection is (0, 2).
The bytes in `beta` after post-selection:
[]
The shape of `beta` after bit-wise slicing is (4096, 1).
The bytes in `beta` after bit-wise slicing:
[[7]
 [7]
 [7]
 ...
 [0]
 [0]
 [7]]

The shape of `beta` after shot-wise slicing is (5, 2).
The bytes in `beta` after shot-wise slicing:
[[  0 135]
 [  0 247]
 [  1 247]
 [  1 128]
 [  1 255]]

Exp. val. for observable `SparsePauliOp(['ZZZZZZZZZ'],
              coeffs=[1.+0.j])` is: 0.068359375
Exp. val. for observable `SparsePauliOp(['IIIIIIIIZ'],
              coeffs=[1.+0.j])` is: 0.06396484375

The shape of the merged results is (4096, 2).
The bytes of the merged results:
[[  1  15]
 [  1 239]
 [  3 239]
 

## Result metadata

In addition to the execution results, both the `PrimitiveResult` and `PubResult` objects contain a metadata attribute about the job that was submitted. The metadata containing information for all submitted PUBs (such as the various [runtime options](/docs/api/qiskit-ibm-runtime/options) available) can be found in the `PrimitiveResult.metatada`, while the metadata specific to each PUB is found in `PubResult.metadata`.

<Admonition type="note">
In the metadata field, primitive implementations can return any information about execution that is relevant to them, and there are no key-value pairs that are guaranteed by the base primitive. Thus, the returned metadata might be different in different primitive implementations.
</Admonition>

In [8]:
# Print out the results metadata
print("The metadata of the PrimitiveResult is:")
for key, val in result.metadata.items():
    print(f"'{key}' : {val},")

print("\nThe metadata of the PubResult result is:")
for key, val in result[0].metadata.items():
    print(f"'{key}' : {val},")

The metadata of the PrimitiveResult is:
'execution' : {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2026-01-15 08:07:33', stop='2026-01-15 08:07:36', size=4096>)])},
'version' : 2,

The metadata of the PubResult result is:
'circuit_metadata' : {},


#### Paano kinakalkula ng Estimator ang error
Bukod sa estimate ng mean ng mga observable na ipinasa sa mga input na PUB (ang field na `evs` ng `DataBin`), sinisikap din ng Estimator na maghatid ng estimate ng error na kaugnay ng mga expectation value na iyon. Lahat ng query sa estimator ay magpapopulate ng field na `stds` ng isang quantity na katulad ng standard error of the mean para sa bawat expectation value, ngunit ang ilang opsyon sa error mitigation ay nagbibigay ng karagdagang impormasyon, tulad ng `ensemble_standard_error`.

Isaalang-alang ang isang observable na $\mathcal{O}$. Sa kawalan ng [ZNE](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne), maaari mong isipin ang bawat shot ng Estimator execution bilang nagbibigay ng isang point estimate ng expectation value na $\langle \mathcal{O} \rangle$. Kung ang mga point estimate ay nasa vector na `Os`, ang halagang ibinalik sa `ensemble_standard_error` ay katumbas ng sumusunod (kung saan ang $\sigma_{\mathcal{O}}$ ay ang [standard deviation ng estimate ng expectation value](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) at ang $N_{shots}$ ay ang bilang ng mga shot):

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{shots}} },$$

na itinuturing ang lahat ng shot bilang bahagi ng isang ensemble. Kung humiling ka ng gate [twirling](/guides/error-mitigation-and-suppression-techniques#pauli-twirling) (`twirling.enable_gates = True`), maaari mong ayusin ang mga point estimate ng $\langle \mathcal{O} \rangle$ sa mga set na nagbabahagi ng isang karaniwang twirl. Tawaging `O_twirls` ang mga set ng mga estimate na ito, at mayroon silang `num_randomizations` (bilang ng mga twirl). Ang `stds` ay ang standard error of the mean ng `O_twirls`, tulad ng

$$\frac{ \sigma_{\mathcal{O}} }{ \sqrt{N_{twirls}} },$$

kung saan ang $\sigma_{\mathcal{O}}$ ay ang standard deviation ng `O_twirls` at ang $N_{twirls}$ ay ang bilang ng mga twirl. Kapag hindi mo pinagana ang twirling, ang `stds` at `ensemble_standard_error` ay magkapantay.

Kung pinagana mo ang ZNE, ang mga `stds` na inilarawan sa itaas ay magiging mga timbang sa isang non-linear regression sa isang extrapolator model. Ang halagang ibinabalik sa `stds` sa kasong ito ay ang uncertainty ng fit model na sinusuri sa noise factor na zero. Kapag may mahinang fit, o malaking uncertainty sa fit, ang naiulat na `stds` ay maaaring maging napakalaki. Kapag pinagana ang ZNE, ang `pub_result.data.evs_noise_factors` at `pub_result.data.stds_noise_factors` ay mapupunan din, upang maaari kang gumawa ng sarili mong extrapolation.

### Output ng Sampler
Kapag matagumpay na natapos ang isang Sampler job, ang ibinalik na [`PrimitiveResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) na object ay naglalaman ng listahan ng mga [`SamplerPubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.SamplerPubResult), isa bawat PUB. Ang mga data bin ng mga `SamplerPubResult` na object na ito ay mga dict-like na object na naglalaman ng isang `BitArray` bawat `ClassicalRegister` sa Circuit.

Ang klase ng `BitArray` ay isang container para sa ordered na shot data. Sa mas detalyadong pagpapaliwanag, iniiimbak nito ang mga na-sample na bitstring bilang bytes sa loob ng isang two-dimensional array. Ang kaliwang pinaka-axis ng array na ito ay tumatakbo sa mga ordered na shot, habang ang kanang pinaka-axis ay tumatakbo sa mga byte.

Bilang unang halimbawa, tingnan natin ang sumusunod na sampung Qubit na Circuit: